In [ ]:
#Importing libraries.
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import seaborn as sns
import scipy.stats as stats
from scipy.stats import pearsonr, spearmanr
from scipy.stats import linregress
import statsmodels.formula.api as smf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer


In [ ]:
#Load dataset and inspect data
csv_df = pd.read_csv('movies.csv')

print(f"The First 5 rows: ")
csv_df.head()

In [ ]:
print("INFORMATION ON THE DATASET : \n")
csv_df.info()

In [ ]:
print("Show the number of empty cells in each column: ")
csv_df.isnull().sum()

In [ ]:
#Drop rows with missing values
csv_df = csv_df.dropna(subset=['votes', 'score', 'writer', 'star', 'company', 'released', 'rating'])

#Remove duplicates
csv_df = csv_df.drop_duplicates()

#Removed the country in removed and formatted the date - mm-dd-yyyy.
csv_df['released'] = csv_df['released'].str.replace(r'\s*\(.*?\)$', '', regex = True).str.strip()
csv_df['released'] = pd.to_datetime(csv_df['released'], format = 'mixed')
csv_df.head()


In [ ]:
#Filled the empty cells in Budget with the mean value,
#while for Runtime, Gross, Country filled with mode value,
#removed movies with 'Not Rated'.

csv_df['budget'] = pd.to_numeric(csv_df['budget'], errors = 'coerce')
mean_budget = csv_df['budget'].mean()
csv_df['budget'].fillna(mean_budget, inplace=True)

csv_df['runtime'].fillna(csv_df['runtime'].mode(), inplace = True)
csv_df['gross'].fillna(csv_df['gross'].mode(), inplace = True)
csv_df['country'].fillna(csv_df['country'].mode(), inplace = True)

#WORK LATER TO REMOVE 'Directors', 'Director', 'Director X'
# csv_df = csv_df[~csv_df['rating'].str.contains('Not Rated', na=False)]
# csv_df = csv_df[~csv_df['director'].str.contains('Directors', na=False)&
#                 csv_df[~csv_df['director'].str.contains('Director', na=False)]&
#                 csv_df[~csv_df['director'].str.contains('Director X', na=False)]]
# csv_df['director']

In [ ]:
#Set upper and lower limits for Gross to remove outliers.

Q1, Q3 = csv_df['gross'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = csv_df[(csv_df['gross'] < lower_bound) | (csv_df['gross'] > upper_bound)]
print(outliers)

csv_df['gross'] = csv_df['gross'].clip(lower = lower_bound, upper = upper_bound)

In [ ]:
#Set upper and lower limits for Gross to remove outliers.
Q1, Q3 = csv_df['budget'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
budget_outliers = csv_df[(csv_df['budget'] < lower_bound) | (csv_df['budget'] > upper_bound)]
print(outliers)

csv_df['budget'] = csv_df['budget'].clip(lower = lower_bound, upper = upper_bound)

csv_df['budget'] = pd.to_numeric(csv_df['budget'], errors = 'coerce')

In [ ]:
#Filter genres with enough data

genre_counts = csv_df['genre'].value_counts()
valid_genres = genre_counts[genre_counts >= 2].index
mov_df = csv_df[csv_df['genre'].isin(valid_genres)]


In [ ]:

genre_counts = mov_df['genre'].value_counts()
most_common = genre_counts.idxmax()
print(f"Most common genre: {most_common} ({genre_counts[most_common]} movies)")

avg_rating = mov_df.groupby('genre')['score'].mean()
highest_rated = avg_rating.idxmax()
print(f"Genre with highest average rating: {highest_rated} ({avg_rating[highest_rated]:.2f})")


In [ ]:
#Distribution of genres
plt.figure(figsize = (10, 6))
genre_counts.plot(kind = 'bar')
plt.yticks(range(5, 2501, 145))
plt.title('Number of Movies Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.show()

#Distribution of genres
plt.figure(figsize = (10, 6))
avg_rating.plot(kind = 'bar')
plt.yticks(range(0, 10, 1))
plt.title('Average Rating by Genre')
plt.xlabel('Genre')
plt.ylabel('Average Rating')
plt.show()

In [ ]:

valid_directors = director_counts[director_counts >= 5].index
mov_df = mov_df[mov_df['director'].isin(valid_directors)]

director_counts = mov_df['director'].value_counts()
top_director = director_counts.idxmax()
print(f"Director with most movies: {top_director} ({director_counts[top_director]} movies)")

overall_mean = mov_df['score'].mean()
director_ratings = mov_df.groupby('director')['score'].mean()
top_director_mean = director_ratings[top_director]
print(f"Overall average rating: {overall_mean:.2f}")
print(f"{top_director}'s average rating: {top_director_mean:.2f}")


top_director_scores = mov_df[mov_df['director'] == top_director]['score']
other_scores = top_director_scores = mov_df[mov_df['director'] != top_director]['score']
t_stat, p_value = stats.ttest_ind(top_director_scores, other_scores, equal_var = False)
print(f"T-test: t={t_stat:.2f}, p-value={p_value:.4f}")
print("Significant if p < 0.05")

#Distribution of genres
plt.figure(figsize = (12, 6))
director_counts.head(15).plot(kind = 'bar', color = 'skyblue')
plt.xticks(rotation = 45)
plt.title('Top 15 Directors by Movies Count')
plt.xlabel('Directors')
plt.ylabel('Number of Movies')
plt.show()

#Distribution of genres
plt.figure(figsize = (12, 12))
top_directors = director_counts.head(10).index
subset = mov_df[mov_df['director'].isin(top_directors)]
subset.boxplot(column = 'score', by = 'director', grid = False)
plt.xticks(rotation = 45)
plt.title('Rating Distribution forTop 10 Directors')
plt.suptitle('')
plt.ylabel('Ratings')
plt.show()


results = pd. DataFrame({
    'Movies': director_counts.values,
    'Avg Ratings': director_ratings.reindex(director_counts.index).round(2)
}).head(15)

results


In [ ]:
mov_df = mov_df[(mov_df['budget'] > 0) & (mov_df['gross'] > 0)]
mov_df['log_budget'] = np.log1p(mov_df['budget'])
mov_df['log_gross'] = np.log1p(mov_df['gross'])

In [148]:
pearson_corr, pearson_p = pearsonr(mov_df['budget'], mov_df['gross'])
log_pearson_corr, log_pearson_p = pearsonr(mov_df['log_budget'], mov_df['log_gross'])
print(f"Pearson (raw): {pearson_corr:.3f}, p={pearson_p:.4f}")
print(f"Pearson (log): {log_pearson_corr:.3f}, p={log_pearson_p:.4f}")


Pearson (raw): 0.735, p=0.0000
Pearson (log): 0.653, p=0.0000


In [149]:
spearman_corr, spearman_p = spearmanr(mov_df['budget'], mov_df['gross'])
print(f"Spearman (raw): {spearman_corr:.3f}, p={spearman_p:.4f}")

Spearman (raw): 0.688, p=0.0000


In [ ]:
plt.figure(figsize = (12, 6))
sns.scatterplot(x = 'log_budget', y = 'log_gross', hue = 'genre', size = 'budget', data = mov_df, alpha = 0.6)
sns.regplot(x = 'log_budget', y = 'log_gross', data = mov_df, scatter = False, color = 'black')
plt.title('Log Budget vs. Log Gross Revenue by Genre')
plt.xlabel('Log Budget ($)')
plt.ylabel('Log Gross Revenue ($)')
plt.show()



In [ ]:
plt.figure(figsize = (12, 6))
plt.subplot(1, 2, 1)
sns.boxplot(y = mov_df['log_budget'])
plt.title('Log Budget Distribution')
plt.subplot(1, 2, 2)
sns.boxplot(y = mov_df['gross'])
plt.title('Revenue Distribution')
plt.show()

In [ ]:
#COME BACK TO WORK ON THIS

# for genre in mov_df['genre'].unique:
#     subset = mov_df[mov_df['genre'] == genre]
#     if len(subset) > 1:
#         corr, p = pearsonr(subset['log_budget'], subset['log_gross'])
#         print(f"{genre}: Log Pearson Corr={corr:.3f}, p={p:.4f}")

In [ ]:
pearsonr(mov_df['budget'] * mov_df['runtime'], mov_df['gross']) #Interaction term

1. How has the average runtime of movies changed over the years?
2. Which country produces the most movies?

First of, I group the dataset by year and country, then compute for averages and count values, and fiinally visualize the trends with plots.

There was a dip in runtime from 1985 to 1988, which might tie to direct-to-video trends - likely due to format shifts(streaming) or budget cuts, requires further analysis
There has been a steady increase in runtime length, with a sharp rise from 2018 to 2020
United States of America is the country with the most movies with 2682 movies

In [ ]:
mov_df['released'] = pd.to_datetime(mov_df['released'], errors = 'coerce')
mov_df['released_year'] = mov_df['released'].dt.year
mov_df = mov_df.dropna(subset = ['released_year', 'runtime', 'country'])
mov_df = mov_df[mov_df['runtime'] > 0]

mov_df.head()

In [ ]:
runtime_by_year = mov_df.groupby('released_year')['runtime'].mean()
print(f"Average runtime trend calculated for {len(runtime_by_year)} years")

In [ ]:
country_counts = mov_df['country'].value_counts()
top_country = country_counts.idxmax()
print(f"Country with most movies: {top_country} ({country_counts[top_country]} movies)")

In [ ]:
plt.figure(figsize = (12, 6))
sns.lineplot(x = runtime_by_year.index, y = runtime_by_year.values, marker = 'o', color = 'teal')
plt.title('Average Movie Runtime Over Years')
plt.xlabel('Year')
plt.ylabel('Average Runtime (minutes)')
plt.grid(True)
plt.show()

In [ ]:
runtime_by_year.rolling(window = 3, center = True).mean().plot()

In [ ]:
plt.figure(figsize = (12, 6))
country_counts.head(15).plot(kind = 'bar', color = 'skyblue')
plt.xticks(rotation = 45)
plt.yticks(range(5, 2950, 145))
plt.title('Top 15 Countries by Movie Count')
plt.xlabel('Country')
plt.ylabel('Number of Movies')
plt.show()

In [ ]:

#Compute regression stats using filtered genre 
results = []
for genre in mov_df['genre'].unique():
    subset = mov_df[ mov_df['genre'] == genre]
    model = smf.ols('gross ~ votes', data = subset).fit()
    results.append({
        'Genre': genre,
        'Slope': round(model.params['votes'], 2),
        'Intercept': round(model.params['Intercept'], 2),
        'R^2': round(model.rsquared, 3) if model.rsquared > 0 else 'N/A',
        'p-value': round(model.pvalues['votes'], 4)
    })
    
results_df = pd.DataFrame(results)
results_df

In [ ]:
#Relevance of Scores and Votes to Gross Revenue
sns.lmplot(x = 'votes', y = 'gross', hue = 'genre', data = mov_df, height = 6, aspect = 1.5)

plt.xlabel('Number of Votes')
plt.ylabel('Gross Revenue ($)')
plt.title('Impact of Score and Votes on Gross Revenue')
plt.show()

In [ ]:
#Budget and Gross correlation
correlation = csv_df['budget'].corr(csv_df['gross'])
print(f"Correlation between the budget and gross income is: {correlation}")
plt.figure(figsize = (14, 8))
csv_df.plot.scatter(x = 'budget', y = 'gross')
plt.title('Budget vs Gross')
plt.show()


In [ ]:
plt.figure(figsize = (10,6))
plt.scatter(csv_df['score'], csv_df['gross'], s = csv_df['votes']/10000, alpha = 0.5)
plt.xlabel('Score')
plt.ylabel('Gross Revenue ($)')
plt.title('Impact of Score and Votes on Gross Revenue')
plt.grid(True)
plt.show()

In [ ]:
#Number of Movies released each Year except year-2020

csv_df['release_year'] = csv_df['released'].dt.year

movies_per_year = csv_df.groupby('release_year').size()[:-1]
plt.figure(figsize = (10, 6))
movies_per_year.plot(kind = 'line')
plt.title('Movies per Year')
plt.xlabel('Year')
plt.ylabel('Number of Movies')
plt.show()

In [ ]:
csv_df = csv_df.drop('release_year', axis = 1)
csv_df = csv_df.dropna()
csv_df.isnull().sum()

In [ ]:
#Defining features
cat_cols = ['genre', 'director', 'country', 'star', 'company', 'writer']
num_cols = ['budget', 'runtime', 'score', 'votes']

In [ ]:
X = csv_df.drop('gross', axis = 1)
y = csv_df['gross']



preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), cat_cols)
    ])


model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state = 42))
])


csv_df.sort_values('released', inplace = True)
train_size = int(0.8*len(csv_df))
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

In [ ]:
#Feature importance
importances = model.named_steps['regressor'].feature_importances_
feature_names = preprocessor.get_feature_names_out()
feature_importance_csv_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
print("\nTop 10 Most Important Features.")
print(feature_importance_csv_df.sort_values('importance', ascending = False).head(10))

In [ ]:
plt.figure(figsize = (10, 6))
plt.scatter(y_test, y_pred, alpha = 0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'g--') #Diagonal line
plt.xlabel('Actual Revenue')
plt.ylabel('Predicted Revenue')
plt.title('Actual vs Predicted Revenue')
plt.show()

In [ ]:
import plotly.express as px
fig = px.scatter(csv_df, x='budget', y='gross', color='genre', title='Budget vs Revenue by Genre')
fig.show()